# Evaluación de Cellpose-SAM sobre el corpus de mitocondrias

**Barrido de hiperparámetros de inferencia** de un modelo pre-entrenado
(Cellpose-SAM), evaluado *zero-shot* contra el ground truth anotado a nivel de
instancia.

> ⚠️ Esto **no es fine-tuning**: no se actualizan los pesos del modelo. Es un
> barrido de parámetros de `model.eval()` (`flow_threshold`,
> `cellprob_threshold`, `niter`). Son cosas distintas en la tesis (ver CLAUDE.md).

## Metodología (según CLAUDE.md)
- Matching uno-a-uno por **algoritmo húngaro** sobre `1 - IoU` (no *greedy*).
- Métricas a nivel instancia con τ **parametrizable**; barrido
  τ ∈ {0.5, 0.55, …, 0.9}, con τ = 0.5 como referencia.
- Se elige **una única configuración global** que maximiza la métrica
  **promediada sobre todas las imágenes** — no un óptimo por-imagen (eso sería
  un oráculo que usa el GT de cada imagen).
- **Una sola** implementación de `iou_matrix`, reutilizada por las métricas y por
  la visualización.


## Errores corregidos respecto del notebook previo

**Bugs de ejecución**
1. `train_with_parameters` usaba variables inexistentes (`masks`, `flows` fuera
   de scope) y pasaba el diccionario `images` a `masks_flows_to_seg` en lugar de
   la imagen. Se elimina esa función rota; el guardado se hace aparte.
2. Fórmulas del markdown de métricas equivocadas: `precision = TP/(TP+FN)` (debía
   ser `TP/(TP+FP)`) y `f1 = 2·P·R/(P·R)` (debía ser `/(P+R)`).

**Errores metodológicos (CLAUDE.md "Errores conocidos a NO repetir")**
3. *Selección de hiperparámetros por imagen = oráculo.* Se reemplaza por elegir
   una única config que maximiza el promedio sobre imágenes.
4. *`iou_matrix` duplicada.* Ahora hay una sola, reutilizada por métricas y
   visualización. La copia de la viz asumía fondo en el primer ID
   (`np.unique(...)[1:]`); se filtra por `!= 0`.
5. *"Fine-tuning" mal nombrado.* Se explicita que es barrido de inferencia.

**Especificación de métricas**
6. `average_precision` de Cellpose (matching *greedy*) ya **no** es la referencia;
   se usa solo para contraste opcional documentado.
7. Se agregan `error_rate = (FP+FN)/(TP+FN)` y `ap_cellpose = TP/(TP+FN+FP)` con
   los nombres correctos; τ es un parámetro y se barre.
8. Instancias GT de área 0 e imágenes con `M = 0` se excluyen del promedio.


In [ ]:
!pip install git+https://www.github.com/mouseland/cellpose.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

from cellpose import io, models
# Se importa SOLO para contraste opcional (matching greedy). Nunca como
# implementación de referencia de las métricas (ver CLAUDE.md).
from cellpose.metrics import average_precision

In [ ]:
# --- Rutas (Colab / Google Drive) ---
BASE_DIR = "/content/drive/MyDrive/Tesis Santiago Feliu/Notebooks/Cellpose"
IMAGES_DIR = f"{BASE_DIR}/Images_ext"
OUTPUT_DIR = f"{BASE_DIR}/predicted_masks_ext"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Grilla de hiperparámetros de INFERENCIA (no fine-tuning) ---
PARAM_GRID = {
    "flow_threshold": [0.2, 0.4, 0.6, 0.8],
    "cellprob_threshold": [-4.0, 0.0, 4.0],
    "niter": [0, 1000, 2000, 4000],
}

# --- Umbrales de IoU para el matching a nivel instancia ---
TAUS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
TAU_REF = 0.5  # valor de referencia

HERRAMIENTA = "cellpose_sam"

# --- Nueve frames de referencia para la visualización cualitativa (A_t1..C_t3) ---
FRAMES_REFERENCIA = {
    "P07T50":  "ImageFV10006.50007_C002T050",
    "P07T164": "ImageFV10006.50007_C002T164",
    "P08T47":  "ImageFV10006.50008_C002T047",
    "P08T121": "ImageFV10006.50008_C002T121",
    "P28T27":  "ImageFV10006.50028_C002T027",
    "P28T177": "ImageFV10006.50028_C002T177",
    "P36T86":  "ImageFV10006.50036_C002T086",
    "P36T118": "ImageFV10006.50036_C002T118",
    "P37T17":  "ImageFV10006.50037_C002T017",
    "P37T100": "ImageFV10006.50037_C002T100",
}

In [ ]:
def get_ground_truth(image_filepath: str) -> dict:
    """Carga el _seg.npy de Cellpose asociado a una imagen .tif."""
    return np.load(image_filepath[:-4] + "_seg.npy", allow_pickle=True).item()


def get_tif_files(directory: str) -> dict:
    """Devuelve {archivo.tif: ruta_completa} para todos los .tif del directorio."""
    return {
        f: os.path.join(directory, f)
        for f in sorted(os.listdir(directory))
        if f.endswith(".tif")
    }

## Métricas a nivel instancia

Entrada: dos máscaras de etiquetas 2D sobre la misma imagen (`0` = fondo,
`1..K` = IDs de instancia).

1. **Matriz IoU** `M × N`, excluyendo el fondo por valor (`!= 0`).
2. **Matching húngaro** sobre `1 - IoU`; se descartan pares con `IoU < τ`.
3. **Conteos:** `TP` = pares que sobreviven; `FN = M − TP`; `FP = N − TP`.
4. **Métricas:**

```
error_rate  = (FP + FN) / (TP + FN)    # denom = nº de instancias GT; no acotado por 1
ap_cellpose = TP / (TP + FN + FP)      # Jaccard de instancias (convención Cellpose)
precision   = TP / (TP + FP)
recall      = TP / (TP + FN)
f1          = 2·P·R / (P + R)
```

`ap_cellpose` **no** es el average precision clásico (área bajo la curva PR): es
un Jaccard a nivel de instancias, la convención de Cellpose. `f1` y `ap_cellpose`
ordenan igual (`f1 = 2·AP/(1+AP)`); `error_rate` aporta información distinta.


In [ ]:
def iou_matrix(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """
    Matriz de IoU entre instancias de dos máscaras de etiquetas 2D.

    Ambas máscaras: 0 = fondo, 1..K = IDs de instancia. El fondo se filtra por
    valor (`!= 0`), NO se asume que sea el primer ID. np.unique sólo devuelve
    etiquetas presentes, por lo que las instancias de área 0 no aparecen.

    Retorna (gt_ids, pr_ids, iou), con iou de forma (len(gt_ids), len(pr_ids)).
    """
    gt_ids = np.unique(gt_lab)
    gt_ids = gt_ids[gt_ids != 0]
    pr_ids = np.unique(pr_lab)
    pr_ids = pr_ids[pr_ids != 0]

    iou = np.zeros((len(gt_ids), len(pr_ids)), dtype=float)
    if len(gt_ids) == 0 or len(pr_ids) == 0:
        return gt_ids, pr_ids, iou

    gt_areas = {i: int((gt_lab == i).sum()) for i in gt_ids}
    pr_areas = {j: int((pr_lab == j).sum()) for j in pr_ids}
    for gi, i in enumerate(gt_ids):
        mask_i = (gt_lab == i)
        for pj, j in enumerate(pr_ids):
            inter = int(np.logical_and(mask_i, pr_lab == j).sum())
            if inter > 0:
                iou[gi, pj] = inter / (gt_areas[i] + pr_areas[j] - inter)
    return gt_ids, pr_ids, iou


def match_instances(gt_lab: np.ndarray, pr_lab: np.ndarray, tau: float):
    """
    Matching uno-a-uno GT<->Pred por algoritmo húngaro sobre 1 - IoU.
    Descarta los pares asignados con IoU < tau.

    Reutiliza iou_matrix (única implementación). Retorna un dict con los conteos
    (tp, fp, fn) y los IDs por categoría, para que la visualización comparta
    exactamente el mismo matching que las métricas.
    """
    gt_ids, pr_ids, iou = iou_matrix(gt_lab, pr_lab)

    matched_gt_ids, matched_pr_ids = [], []
    if iou.size > 0:
        gi, pj = linear_sum_assignment(1.0 - iou)
        for g, p in zip(gi, pj):
            if iou[g, p] >= tau:
                matched_gt_ids.append(gt_ids[g])
                matched_pr_ids.append(pr_ids[p])

    tp = len(matched_pr_ids)
    fp = len(pr_ids) - tp
    fn = len(gt_ids) - tp
    return {
        "tp": tp, "fp": fp, "fn": fn,
        "tp_pr_ids": set(matched_pr_ids),
        "fp_pr_ids": set(pr_ids) - set(matched_pr_ids),
        "fn_gt_ids": set(gt_ids) - set(matched_gt_ids),
        "n_gt": len(gt_ids), "n_pr": len(pr_ids),
    }


def instance_metrics(gt_lab: np.ndarray, pr_lab: np.ndarray, tau: float = TAU_REF):
    """
    Métricas a nivel instancia para un τ dado (ver CLAUDE.md).

    Si no hay instancias en el GT (M = 0), retorna None para excluir la imagen
    del promedio (denominador cero).
    """
    m = match_instances(gt_lab, pr_lab, tau)
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    if (tp + fn) == 0:  # M = 0: imagen sin GT -> excluida del promedio
        return None

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    error_rate = (fp + fn) / (tp + fn)
    ap_cellpose = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else 0.0
    return {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "error_rate": error_rate, "ap_cellpose": ap_cellpose,
    }

## Métricas pixel a pixel (como Ilastik)

Las herramientas de instancias también se evalúan a nivel **píxel**: binarizando
la máscara (`label > 0` → primer plano) y aplicando la misma matriz de confusión
foreground/fondo que usa Ilastik. **No** depende de τ (τ es del matching de
instancias); se calcula una vez por predicción.


In [ ]:
from matplotlib.colors import ListedColormap

_PX_CMAP = ListedColormap(["black", "white", "green", "red"])  # TN, TP, FN, FP


def pixel_metrics(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """
    Métricas pixel a pixel (foreground vs fondo), misma convención que Ilastik
    (calculate_segmentation_metrics). Binariza label > 0. Columnas con prefijo px_.
    """
    y_true = gt_lab > 0
    y_pred = pr_lab > 0
    tp = int(np.sum(y_true & y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return {
        "px_tp": tp, "px_tn": tn, "px_fp": fp, "px_fn": fn,
        "px_precision": precision, "px_recall": recall, "px_f1": f1,
        "px_iou": iou, "px_specificity": specificity, "px_accuracy": accuracy,
        "px_dice": f1,
    }


def pixel_superposition(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """Mapa de superposición pixel a pixel (estilo Ilastik):
    0 = fondo (TN, negro), 1 = TP (blanco), 2 = FN (verde), 3 = FP (rojo)."""
    y_true = gt_lab > 0
    y_pred = pr_lab > 0
    sup = np.zeros(gt_lab.shape, dtype=np.uint8)
    sup[y_true & y_pred] = 1
    sup[y_true & ~y_pred] = 2
    sup[~y_true & y_pred] = 3
    return sup

## Barrido de la grilla sobre TODAS las imágenes

Se evalúa cada configuración de la grilla en cada imagen. Se calculan las métricas
de **instancia** por cada τ y las de **píxel** una sola vez por config. **No** se
elige el mejor parámetro por imagen (oráculo). La configuración global se elige
después, promediando sobre imágenes.


In [ ]:
def evaluar_grilla(param_grid: dict, images: dict, taus=TAUS, model=None):
    """
    Corre el modelo sobre TODAS las imágenes y toda la grilla. Devuelve DOS
    DataFrames:
      - instancia: una fila por (imagen, config, τ)
      - píxel:     una fila por (imagen, config)  [no depende de τ]

    Cada imagen se infiere UNA sola vez por configuración.
    """
    if model is None:
        model = models.CellposeModel(gpu=True)

    combos = list(itertools.product(
        param_grid["flow_threshold"],
        param_grid["cellprob_threshold"],
        param_grid["niter"],
    ))

    filas_inst, filas_px = [], []
    for image_name, image_filepath in images.items():
        print(f"Procesando {image_name}  ({len(combos)} configs x {len(taus)} τ)")
        gt_lab = get_ground_truth(image_filepath)["masks"]
        img = io.imread(image_filepath)

        for flow, cellprob, niter in combos:
            pr_lab, _, _ = model.eval(
                img, flow_threshold=flow, cellprob_threshold=cellprob, niter=niter
            )
            cfg_cols = {
                "image_name": image_name,
                "herramienta": HERRAMIENTA,
                "flow_threshold": flow,
                "cellprob_threshold": cellprob,
                "niter": niter,
            }
            # Píxel: una sola vez por config (no depende de τ)
            filas_px.append({**cfg_cols, **pixel_metrics(gt_lab, pr_lab)})
            # Instancia: por cada τ
            for tau in taus:
                met = instance_metrics(gt_lab, pr_lab, tau=tau)
                if met is None:  # imagen sin GT -> excluida
                    continue
                filas_inst.append({**cfg_cols, "tau": tau, **met})
    return pd.DataFrame(filas_inst), pd.DataFrame(filas_px)

In [ ]:
model = models.CellposeModel(gpu=True)
resultados, resultados_px = evaluar_grilla(PARAM_GRID, get_tif_files(IMAGES_DIR), model=model)
resultados.to_csv(f"{OUTPUT_DIR}/resultados_por_imagen.csv", index=False)
resultados_px.to_csv(f"{OUTPUT_DIR}/resultados_pixel_por_imagen.csv", index=False)
print("instancia:", resultados.shape, "| pixel:", resultados_px.shape)
resultados_px.head()

## Selección de la configuración global

Para un τ fijo (referencia = 0.5), se promedia cada métrica sobre las imágenes
para cada configuración y se elige la que maximiza el promedio. **Una** sola
configuración para toda la herramienta, no una por imagen.


In [ ]:
def seleccionar_config_global(resultados: pd.DataFrame, tau=TAU_REF, metrica="f1"):
    """
    Promedia la métrica sobre imágenes para cada configuración (a un τ fijo) y
    devuelve (fila_ganadora, tabla_ordenada). Forma correcta de elegir: una única
    config que maximiza la media sobre imágenes, no un óptimo por-imagen.
    """
    cols_cfg = ["flow_threshold", "cellprob_threshold", "niter"]
    cols_met = ["precision", "recall", "f1", "error_rate", "ap_cellpose"]
    agg = (
        resultados[resultados["tau"] == tau]
        .groupby(cols_cfg)[cols_met]
        .mean()
        .reset_index()
        .sort_values(metrica, ascending=False)
        .reset_index(drop=True)
    )
    return agg.iloc[0], agg


mejor_config, tabla_configs = seleccionar_config_global(resultados, tau=TAU_REF, metrica="f1")
tabla_configs.to_csv(f"{OUTPUT_DIR}/config_ranking_tau{TAU_REF}.csv", index=False)
print(f"Mejor configuración global (τ={TAU_REF}, por f1 promedio sobre imágenes):")
print(mejor_config)
tabla_configs.head(10)

In [ ]:
# Config global elegida por instancia (para reusar en curvas y visualización)
cfg = {
    "flow_threshold": float(mejor_config["flow_threshold"]),
    "cellprob_threshold": float(mejor_config["cellprob_threshold"]),
    "niter": int(mejor_config["niter"]),
}
cfg

## Selección de configuración por píxel y comparación

Se elige también la mejor config a nivel **píxel** (por `px_f1`, promedio sobre
imágenes) y se la compara con la mejor por **instancia**. Pueden diferir: la de
píxel favorece cubrir bien el área, la de instancia favorece separar objetos.


In [ ]:
def seleccionar_config_pixel(resultados_px: pd.DataFrame, metrica="px_f1"):
    """Promedia las px-métricas sobre imágenes por config y elige la mejor."""
    cols_cfg = ["flow_threshold", "cellprob_threshold", "niter"]
    cols_met = ["px_precision", "px_recall", "px_f1", "px_iou", "px_specificity", "px_accuracy"]
    agg = (
        resultados_px.groupby(cols_cfg)[cols_met]
        .mean()
        .reset_index()
        .sort_values(metrica, ascending=False)
        .reset_index(drop=True)
    )
    return agg.iloc[0], agg


mejor_config_px, tabla_configs_px = seleccionar_config_pixel(resultados_px, metrica="px_f1")
tabla_configs_px.to_csv(f"{OUTPUT_DIR}/config_ranking_pixel.csv", index=False)
print("Mejor configuración global por px_f1 (promedio sobre imágenes):")
print(mejor_config_px)
tabla_configs_px.head(10)

In [ ]:
cfg_px = {
    "flow_threshold": float(mejor_config_px["flow_threshold"]),
    "cellprob_threshold": float(mejor_config_px["cellprob_threshold"]),
    "niter": int(mejor_config_px["niter"]),
}
cfg_px

In [ ]:
# Comparación: config mejor-por-instancia vs mejor-por-píxel, con métricas de AMBOS niveles.
def _inst_f1(c, tau=TAU_REF):
    m = (
        (resultados["tau"] == tau) &
        (resultados["flow_threshold"] == c["flow_threshold"]) &
        (resultados["cellprob_threshold"] == c["cellprob_threshold"]) &
        (resultados["niter"] == c["niter"])
    )
    return resultados[m]["f1"].mean()


def _px_f1(c):
    m = (
        (resultados_px["flow_threshold"] == c["flow_threshold"]) &
        (resultados_px["cellprob_threshold"] == c["cellprob_threshold"]) &
        (resultados_px["niter"] == c["niter"])
    )
    return resultados_px[m]["px_f1"].mean()


comparacion = pd.DataFrame([
    {"seleccion": "instancia (f1)", **cfg,    "f1_instancia": _inst_f1(cfg),    "px_f1": _px_f1(cfg)},
    {"seleccion": "pixel (px_f1)",  **cfg_px, "f1_instancia": _inst_f1(cfg_px), "px_f1": _px_f1(cfg_px)},
])
comparacion.to_csv(f"{OUTPUT_DIR}/comparacion_instancia_vs_pixel.csv", index=False)
print("Comparación config instancia vs píxel:")
comparacion

## Curvas de métricas vs τ (config global)

Para la configuración elegida, se promedian las métricas sobre imágenes a lo
largo del barrido de τ.


In [ ]:
mask_cfg = (
    (resultados["flow_threshold"] == cfg["flow_threshold"]) &
    (resultados["cellprob_threshold"] == cfg["cellprob_threshold"]) &
    (resultados["niter"] == cfg["niter"])
)
curva = (
    resultados[mask_cfg]
    .groupby("tau")[["precision", "recall", "f1", "error_rate", "ap_cellpose"]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
for met in ["precision", "recall", "f1", "ap_cellpose"]:
    plt.plot(curva["tau"], curva[met], marker="o", label=met)
plt.xlabel("τ (umbral de IoU)")
plt.ylabel("valor (promedio sobre imágenes)")
plt.title(f"Métricas vs τ — {cfg}")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()
curva

## Visualización cualitativa de errores

Overlay sobre la imagen original: **verde = TP**, **rojo = FP**, **azul = FN**.
Usa `match_instances`, es decir el mismo matching húngaro que las métricas.
Restringida a los nueve frames de referencia (`A_t1` … `C_t3`).


In [ ]:
def visualizar_prediccion(img, gt_lab, pr_lab, tau=TAU_REF, title=None, ax=None):
    """
    Overlay de errores: verde = TP (pred emparejada), rojo = FP (pred sin match),
    azul = FN (instancia GT sin match). Reutiliza match_instances.
    """
    m = match_instances(gt_lab, pr_lab, tau)
    overlay = np.zeros((*gt_lab.shape, 3), dtype=np.float32)
    for p in m["tp_pr_ids"]:
        overlay[pr_lab == p] = [0, 1, 0]
    for p in m["fp_pr_ids"]:
        overlay[pr_lab == p] = [1, 0, 0]
    for g in m["fn_gt_ids"]:
        overlay[gt_lab == g] = [0, 0, 1]

    creado = ax is None
    if creado:
        _, ax = plt.subplots(figsize=(6, 6))
    im = img.astype(float)
    if im.max() > im.min():
        im = (im - im.min()) / (im.max() - im.min())
    ax.imshow(im, cmap="gray")
    ax.imshow(overlay, alpha=0.4)
    ax.set_title(title or f"TP (verde) / FP (rojo) / FN (azul) — τ={tau}")
    ax.axis("off")
    if creado:
        plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for ax, (label, stem) in zip(axes.ravel(), FRAMES_REFERENCIA.items()):
    image_filepath = f"{IMAGES_DIR}/{stem}.tif"
    gt_lab = get_ground_truth(image_filepath)["masks"]
    img = io.imread(image_filepath)
    pr_lab, _, _ = model.eval(
        img,
        flow_threshold=cfg["flow_threshold"],
        cellprob_threshold=cfg["cellprob_threshold"],
        niter=cfg["niter"],
    )
    visualizar_prediccion(img, gt_lab, pr_lab, tau=TAU_REF, title=label, ax=ax)

plt.suptitle(f"Cellpose-SAM — config global {cfg} — τ={TAU_REF}", fontsize=16)
plt.tight_layout()
plt.show()

## Superposición pixel a pixel (estilo Ilastik)

Mapa por píxel: **negro = fondo**, **blanco = TP**, **verde = FN**, **rojo = FP**.
Se grafica para la mejor config por **instancia** y por **píxel**, para compararlas.


In [ ]:
def grilla_superposicion_pixel(cfg, titulo):
    """Grilla 3x3 de superposición pixel a pixel sobre los frames de referencia."""
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    for ax, (label, stem) in zip(axes.ravel(), FRAMES_REFERENCIA.items()):
        image_filepath = f"{IMAGES_DIR}/{stem}.tif"
        gt_lab = get_ground_truth(image_filepath)["masks"]
        img = io.imread(image_filepath)
        pr_lab, _, _ = model.eval(
            img,
            flow_threshold=cfg["flow_threshold"],
            cellprob_threshold=cfg["cellprob_threshold"],
            niter=cfg["niter"],
        )
        sup = pixel_superposition(gt_lab, pr_lab)
        ax.imshow(sup, cmap=_PX_CMAP, vmin=0, vmax=3, interpolation="nearest")
        ax.set_title(label)
        ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["black", "white", "green", "red"]]
    fig.legend(handles, ["Fondo", "TP", "FN", "FP"], loc="lower center", ncol=4, fontsize=12)
    plt.suptitle(titulo, fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.06)
    plt.show()


grilla_superposicion_pixel(cfg, f"Superposición píxel — config instancia {cfg}")
grilla_superposicion_pixel(cfg_px, f"Superposición píxel — config píxel {cfg_px}")

## (Opcional) Contraste con `cellpose.metrics.average_precision`

`average_precision` de Cellpose usa matching **greedy**, no húngaro; por eso **no**
es la implementación de referencia. Se la usa sólo para detectar discrepancias
con la implementación propia y documentarlas.


In [ ]:
def contrastar_con_cellpose(gt_lab, pr_lab, tau=TAU_REF):
    """Compara la métrica propia (húngara) contra average_precision (greedy)."""
    ap, tp, fp, fn = average_precision(gt_lab, pr_lab, threshold=[tau])
    propia = instance_metrics(gt_lab, pr_lab, tau=tau)
    return {
        "greedy_ap_cellpose": float(ap[0]),
        "greedy_tp": int(tp[0]), "greedy_fp": int(fp[0]), "greedy_fn": int(fn[0]),
        "hungaro_ap_cellpose": propia["ap_cellpose"],
        "hungaro_tp": propia["tp"], "hungaro_fp": propia["fp"], "hungaro_fn": propia["fn"],
    }


_fp = f"{IMAGES_DIR}/{next(iter(FRAMES_REFERENCIA.values()))}.tif"
_gt = get_ground_truth(_fp)["masks"]
_img = io.imread(_fp)
_pr, _, _ = model.eval(
    _img,
    flow_threshold=cfg["flow_threshold"],
    cellprob_threshold=cfg["cellprob_threshold"],
    niter=cfg["niter"],
)
contrastar_con_cellpose(_gt, _pr, tau=TAU_REF)

In [ ]:
# --- Guardado de máscaras predichas (camada de extensión) ---
# Genera y guarda la segmentación de cada frame nuevo para la config global de
# instancia y la de píxel, en subcarpetas de OUTPUT_DIR (formato _pred.npy).
import os as _os
for _nombre, _c in [
    (f"inst_ft{cfg['flow_threshold']}_ct{cfg['cellprob_threshold']}_ni{cfg['niter']}", cfg),
    (f"px_ft{cfg_px['flow_threshold']}_ct{cfg_px['cellprob_threshold']}_ni{cfg_px['niter']}", cfg_px),
]:
    _dir = _os.path.join(OUTPUT_DIR, "masks_" + _nombre)
    _os.makedirs(_dir, exist_ok=True)
    for _name, _fp in get_tif_files(IMAGES_DIR).items():
        _img = io.imread(_fp)
        _pr, _, _ = model.eval(_img, flow_threshold=_c["flow_threshold"],
                               cellprob_threshold=_c["cellprob_threshold"], niter=_c["niter"])
        np.save(_os.path.join(_dir, _name[:-4] + "_pred.npy"), _pr)
    print("Guardadas máscaras en", _dir)
print("Listo: máscaras de extensión guardadas en", OUTPUT_DIR)